# PostHarvest IQ
# Data Cleaning and Feature Engineering

In [1]:
# Start coding here
%pip install pandas sqlalchemy pymysql scikit-learn python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [1]:
#import necessary packages, create and load database
import pandas as pd
import numpy as np
import pickle
import os
from sqlalchemy import create_engine
from dotenv import load_dotenv
from sklearn.preprocessing import MinMaxScaler

load_dotenv()

DB_USER = os.getenv("DB_USER", "root")
DB_PASSWORD = os.getenv("DB_PASSWORD", "")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "3306")
DB_NAME = os.getenv("DB_NAME", "postharvestiq")

engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")
print("Database connection established.")

Database connection established.


In [2]:
#showing the tables
tables = pd.read_sql("SHOW TABLES", engine)
print(tables)

  Tables_in_postharvest_iq
0      fao_producer_prices
1     ghana_exchange_rates
2              wfp_markets
3               wfp_prices


In [3]:
#load all 4 dataset
df_prices  = pd.read_sql("SELECT * FROM wfp_prices", engine)
df_markets = pd.read_sql("SELECT * FROM wfp_markets", engine)
df_fx      = pd.read_sql("SELECT * FROM ghana_exchange_rates", engine)
df_prod    = pd.read_sql("SELECT * FROM fao_producer_prices", engine)

# Standardise column names
df_prices.columns  = df_prices.columns.str.lower().str.strip()
df_markets.columns = df_markets.columns.str.lower().str.strip()

print(f"✓ wfp_prices:          {len(df_prices)} rows")
print(f"✓ wfp_markets:         {len(df_markets)} rows")
print(f"✓ ghana_exchange_rates:{len(df_fx)} rows")
print(f"✓ fao_producer_prices: {len(df_prod)} rows")

✓ wfp_prices:          27636 rows
✓ wfp_markets:         93 rows
✓ ghana_exchange_rates:785 rows
✓ fao_producer_prices: 5696 rows


In [4]:
# See all Northern/Upper region markets so we use exact spellings
northern = df_markets[
    df_markets['admin1'].str.contains('NORTHERN|UPPER', case=False, na=False)
]
print("Northern Ghana markets in registry:")
print(northern[['market_id', 'market', 'admin1', 'admin2', 'latitude', 'longitude']])

print("\nAll unique markets in wfp_prices:")
print(sorted(df_prices['market'].unique()))

Northern Ghana markets in registry:
    market_id         market      admin1                  admin2  latitude  \
11        220         Tamale    NORTHERN  TAMALE NORTH SUB METRO      9.40   
12        221          Bolga  UPPER EAST    BOLGATANGA MUNICIPAL     10.79   
13        222             Wa  UPPER WEST            WA MUNICIPAL     10.05   
16       1712          Banda    NORTHERN                 KPANDAI      8.32   
17       1713          Bawku  UPPER EAST         BAWKU MUNICIPAL     11.05   
18       1714       Bimbilla    NORTHERN           NANUMBA NORTH      8.85   
19       1715          Bongo  UPPER EAST                   BONGO     10.91   
21       1717      Bunkprugu    NORTHERN        BUNKPURUGU YONYO     10.51   
23       1719        Funbisi  UPPER EAST            BUILSA SOUTH     10.45   
24       1720           Garu  UPPER EAST            GARU TEMPANE     10.85   
25       1721        Gushegu    NORTHERN                GUSHIEGU      9.92   
26       1722         Jirapa

In [5]:
#filter to target markets and commodities
TARGET_MARKETS     = ['Tamale', 'Bolgatanga', 'Wa', 'Kumasi', 'Techiman']
TARGET_COMMODITIES = ['Maize', 'Millet', 'Sorghum']

df_prices_filtered = df_prices[
    df_prices['market'].isin(TARGET_MARKETS) &
    df_prices['commodity'].isin(TARGET_COMMODITIES)
].copy()

df_prices_filtered['date'] = pd.to_datetime(df_prices_filtered['date'])

print(f"✓ Filtered price rows: {len(df_prices_filtered)}")
print("\nMarket counts:")
print(df_prices_filtered['market'].value_counts())
print("\nCommodity counts:")
print(df_prices_filtered['commodity'].value_counts())

✓ Filtered price rows: 1937

Market counts:
market
Techiman    576
Tamale      569
Wa          438
Kumasi      354
Name: count, dtype: int64

Commodity counts:
commodity
Millet     681
Maize      651
Sorghum    605
Name: count, dtype: int64


In [6]:
#Merge Dataset-wfp_markets (adds GPS coordinates)
df_markets_clean = df_markets[
    ['market_id', 'latitude', 'longitude', 'admin1', 'admin2']
].copy()
df_markets_clean.columns = [
    'market_id', 'lat_registry', 'lng_registry', 'admin1_registry', 'admin2_registry'
]

df_merged = df_prices_filtered.merge(df_markets_clean, on='market_id', how='left')

# Use registry coordinates (more reliable than wfp_prices lat/lng)
df_merged['latitude']  = df_merged['lat_registry'].fillna(df_merged['latitude'])
df_merged['longitude'] = df_merged['lng_registry'].fillna(df_merged['longitude'])
df_merged.drop(columns=['lat_registry', 'lng_registry'], inplace=True)

df_merged['year_month'] = df_merged['date'].dt.to_period('M')

print(f"✓ After wfp_markets merge: {len(df_merged)} rows")
print(f"  GPS nulls — latitude: {df_merged['latitude'].isna().sum()}, longitude: {df_merged['longitude'].isna().sum()}")
print(df_merged[['date', 'market', 'commodity', 'price', 'latitude', 'longitude']].head(5))

✓ After wfp_markets merge: 1937 rows
  GPS nulls — latitude: 0, longitude: 0
        date    market commodity  price  latitude  longitude
0 2006-01-15    Kumasi     Maize  25.40      6.68      -1.62
1 2006-01-15    Kumasi   Sorghum  38.25      6.68      -1.62
2 2006-01-15  Techiman     Maize  16.15      7.58      -1.93
3 2006-01-15  Techiman   Sorghum  34.50      7.58      -1.93
4 2006-01-15    Tamale     Maize  20.75      9.40      -0.83


In [7]:
#Merge Dataset 3: ghana_exchange_rates
# Remove annual summary rows, keep monthly only
df_fx_monthly = df_fx[df_fx['Months'] != 'Annual value'].copy()
df_fx_monthly['date'] = pd.to_datetime(df_fx_monthly['StartDate'])
df_fx_monthly['year_month'] = df_fx_monthly['date'].dt.to_period('M')

# One exchange rate per month (average if duplicates)
df_fx_monthly = df_fx_monthly.groupby('year_month')['Value'].mean().reset_index()
df_fx_monthly.columns = ['year_month', 'exchange_rate']

df_merged = df_merged.merge(df_fx_monthly, on='year_month', how='left')

print(f"✓ After exchange rate merge: {len(df_merged)} rows")
print(f"  Exchange rate nulls: {df_merged['exchange_rate'].isna().sum()}")
print(df_merged[['date', 'market', 'commodity', 'price', 'exchange_rate']].head(5))

✓ After exchange rate merge: 1937 rows
  Exchange rate nulls: 0
        date    market commodity  price  exchange_rate
0 2006-01-15    Kumasi     Maize  25.40       0.908154
1 2006-01-15    Kumasi   Sorghum  38.25       0.908154
2 2006-01-15  Techiman     Maize  16.15       0.908154
3 2006-01-15  Techiman   Sorghum  34.50       0.908154
4 2006-01-15    Tamale     Maize  20.75       0.908154


In [8]:
#Merge Dataset 4: fao_producer_prices
# Filter to cereals only, annual values only
CEREAL_ITEMS = ['Maize', 'Millet', 'Sorghum']

df_prod_cereals = df_prod[
    df_prod['Item'].isin(CEREAL_ITEMS) &
    (df_prod['Months'] == 'Annual value')
].copy()

print("Cereal items found in producer prices:")
print(df_prod_cereals['Item'].value_counts())

# Average producer price index per year across cereals
df_prod_annual = df_prod_cereals.groupby('Year')['Value'].mean().reset_index()
df_prod_annual.columns = ['year', 'producer_price_index']
df_prod_annual['year'] = df_prod_annual['year'].astype(int)

# Merge on year
df_merged['year'] = df_merged['date'].dt.year
df_merged = df_merged.merge(df_prod_annual, on='year', how='left')

# Calculate price spread: market price minus producer price index
df_merged['price_spread'] = df_merged['price'] - df_merged['producer_price_index']

print(f"\n✓ After producer price merge: {len(df_merged)} rows")
print(f"  Producer price nulls: {df_merged['producer_price_index'].isna().sum()}")
print(f"  Price spread nulls:   {df_merged['price_spread'].isna().sum()}")
print(df_merged[['date', 'market', 'commodity', 'price', 'producer_price_index', 'price_spread']].head(5))

Cereal items found in producer prices:
Item
Millet     119
Sorghum    119
Name: count, dtype: int64

✓ After producer price merge: 1937 rows
  Producer price nulls: 0
  Price spread nulls:   0
        date    market commodity  price  producer_price_index  price_spread
0 2006-01-15    Kumasi     Maize  25.40          959831.84375 -959806.44375
1 2006-01-15    Kumasi   Sorghum  38.25          959831.84375 -959793.59375
2 2006-01-15  Techiman     Maize  16.15          959831.84375 -959815.69375
3 2006-01-15  Techiman   Sorghum  34.50          959831.84375 -959797.34375
4 2006-01-15    Tamale     Maize  20.75          959831.84375 -959811.09375


In [9]:
#Remove Outliers (IQR method, multiplier = 3)
def remove_outliers_iqr(df, price_col='price', multiplier=3):
    clean_frames = []
    for (market, commodity), group in df.groupby(['market', 'commodity']):
        Q1 = group[price_col].quantile(0.25)
        Q3 = group[price_col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - multiplier * IQR
        upper = Q3 + multiplier * IQR
        filtered = group[
            (group[price_col] >= lower) &
            (group[price_col] <= upper)
        ]
        removed = len(group) - len(filtered)
        if removed > 0:
            print(f"  {market} {commodity}: removed {removed} outlier rows")
        clean_frames.append(filtered)
    return pd.concat(clean_frames, ignore_index=True)

df_clean = remove_outliers_iqr(df_merged, price_col='price', multiplier=3)

print(f"\n✓ Before: {len(df_merged)} rows → After outlier removal: {len(df_clean)} rows")

  Kumasi Maize: removed 4 outlier rows
  Kumasi Millet: removed 3 outlier rows
  Kumasi Sorghum: removed 2 outlier rows
  Tamale Maize: removed 4 outlier rows
  Tamale Millet: removed 7 outlier rows
  Tamale Sorghum: removed 9 outlier rows
  Techiman Maize: removed 7 outlier rows
  Techiman Millet: removed 9 outlier rows
  Techiman Sorghum: removed 6 outlier rows
  Wa Maize: removed 12 outlier rows
  Wa Millet: removed 8 outlier rows
  Wa Sorghum: removed 4 outlier rows

✓ Before: 1937 rows → After outlier removal: 1862 rows


In [10]:
#Normalise Price per Market per Commodity
scalers = {}
df_clean['price_scaled'] = np.nan

for (market, commodity), group in df_clean.groupby(['market', 'commodity']):
    scaler = MinMaxScaler()
    idx = group.index
    df_clean.loc[idx, 'price_scaled'] = scaler.fit_transform(
        group[['price']]
    ).flatten()
    key = f"{market}_{commodity}_price"
    scalers[key] = scaler
    print(f"  Scaler saved: {key}")

print(f"\n✓ Total price scalers: {len(scalers)}")

  Scaler saved: Kumasi_Maize_price
  Scaler saved: Kumasi_Millet_price
  Scaler saved: Kumasi_Sorghum_price
  Scaler saved: Tamale_Maize_price
  Scaler saved: Tamale_Millet_price
  Scaler saved: Tamale_Sorghum_price
  Scaler saved: Techiman_Maize_price
  Scaler saved: Techiman_Millet_price
  Scaler saved: Techiman_Sorghum_price
  Scaler saved: Wa_Maize_price
  Scaler saved: Wa_Millet_price
  Scaler saved: Wa_Sorghum_price

✓ Total price scalers: 12


In [11]:
#Add Seasonality Features (month_sin, month_cos)
df_clean['month']     = df_clean['date'].dt.month
df_clean['month_sin'] = np.sin(2 * np.pi * df_clean['month'] / 12)
df_clean['month_cos'] = np.cos(2 * np.pi * df_clean['month'] / 12)

print("✓ Seasonality features added")
print(df_clean[['date', 'month', 'month_sin', 'month_cos']].head(6))

✓ Seasonality features added
        date  month     month_sin  month_cos
0 2006-01-15      1  5.000000e-01   0.866025
1 2006-04-15      4  8.660254e-01  -0.500000
2 2006-05-15      5  5.000000e-01  -0.866025
3 2006-06-15      6  1.224647e-16  -1.000000
4 2006-07-15      7 -5.000000e-01  -0.866025
5 2006-10-15     10 -8.660254e-01   0.500000


In [12]:
#Normalise Exchange Rate and Price Spread Globally
# Fill any remaining nulls before scaling
df_clean['exchange_rate']      = df_clean['exchange_rate'].fillna(df_clean['exchange_rate'].median())
df_clean['price_spread']       = df_clean['price_spread'].fillna(df_clean['price_spread'].median())
df_clean['producer_price_index'] = df_clean['producer_price_index'].fillna(df_clean['producer_price_index'].median())

fx_scaler     = MinMaxScaler()
spread_scaler = MinMaxScaler()

df_clean['fx_scaled']     = fx_scaler.fit_transform(df_clean[['exchange_rate']]).flatten()
df_clean['spread_scaled'] = spread_scaler.fit_transform(df_clean[['price_spread']]).flatten()

scalers['exchange_rate_global'] = fx_scaler
scalers['price_spread_global']  = spread_scaler

print(f"✓ Global scalers added. Total scalers: {len(scalers)}")
print(df_clean[['exchange_rate', 'fx_scaled', 'price_spread', 'spread_scaled']].describe().round(4))

✓ Global scalers added. Total scalers: 14
       exchange_rate  fx_scaled  price_spread  spread_scaled
count      1862.0000  1862.0000     1862.0000      1862.0000
mean          3.8206     0.2394   -98732.2933         0.8967
std           2.5896     0.2129   286666.6317         0.2985
min           0.9082     0.0000  -959820.8438         0.0000
25%           1.4472     0.0443    -1506.7075         0.9979
50%           3.8125     0.2387     -664.5362         0.9988
75%           5.6865     0.3928     -403.8850         0.9991
max          13.0730     1.0000      472.5050         1.0000


In [13]:
os.makedirs('app/ml/models', exist_ok=True)

with open('app/ml/models/price_scalers.pkl', 'wb') as f:
    pickle.dump(scalers, f)

# Verify it saved correctly
with open('app/ml/models/price_scalers.pkl', 'rb') as f:
    verify = pickle.load(f)

print(f"✓ Saved {len(verify)} scalers to app/ml/models/price_scalers.pkl")
print("  Scaler keys:", list(verify.keys()))

✓ Saved 14 scalers to app/ml/models/price_scalers.pkl
  Scaler keys: ['Kumasi_Maize_price', 'Kumasi_Millet_price', 'Kumasi_Sorghum_price', 'Tamale_Maize_price', 'Tamale_Millet_price', 'Tamale_Sorghum_price', 'Techiman_Maize_price', 'Techiman_Millet_price', 'Techiman_Sorghum_price', 'Wa_Maize_price', 'Wa_Millet_price', 'Wa_Sorghum_price', 'exchange_rate_global', 'price_spread_global']


In [14]:
os.makedirs('data/processed', exist_ok=True)

df_final = df_clean.sort_values(
    ['market', 'commodity', 'date']
).reset_index(drop=True)

df_final.to_csv('data/processed/cereals_merged_clean.csv', index=False)

print(f"✓ Saved to data/processed/cereals_merged_clean.csv")
print(f"  Shape: {df_final.shape}")
print(f"  Columns: {df_final.columns.tolist()}")

✓ Saved to data/processed/cereals_merged_clean.csv
  Shape: (1862, 29)
  Columns: ['date', 'admin1', 'admin2', 'market', 'market_id', 'latitude', 'longitude', 'category', 'commodity', 'commodity_id', 'unit', 'priceflag', 'pricetype', 'currency', 'price', 'usdprice', 'admin1_registry', 'admin2_registry', 'year_month', 'exchange_rate', 'year', 'producer_price_index', 'price_spread', 'price_scaled', 'month', 'month_sin', 'month_cos', 'fx_scaled', 'spread_scaled']


In [15]:
print("=" * 55)
print("FINAL SUMMARY — Task 2 Complete")
print("=" * 55)
print(f"Total rows:       {len(df_final)}")
print(f"Total columns:    {len(df_final.columns)}")
print(f"Markets:          {df_final['market'].unique().tolist()}")
print(f"Commodities:      {df_final['commodity'].unique().tolist()}")
print(f"Date range:       {df_final['date'].min().date()} → {df_final['date'].max().date()}")
print(f"Datasets merged:  wfp_prices + wfp_markets + ghana_exchange_rates + fao_producer_prices")
print(f"Scalers saved:    {len(scalers)} (price per market/commodity + fx + spread)")

missing = df_final.isnull().sum()
missing_cols = missing[missing > 0]
if len(missing_cols) == 0:
    print("Missing values:   ✓ Zero missing values confirmed")
else:
    print(f"Missing values:\n{missing_cols}")
print("=" * 55)

FINAL SUMMARY — Task 2 Complete
Total rows:       1862
Total columns:    29
Markets:          ['Kumasi', 'Tamale', 'Techiman', 'Wa']
Commodities:      ['Maize', 'Millet', 'Sorghum']
Date range:       2006-01-15 → 2023-07-15
Datasets merged:  wfp_prices + wfp_markets + ghana_exchange_rates + fao_producer_prices
Scalers saved:    14 (price per market/commodity + fx + spread)
Missing values:   ✓ Zero missing values confirmed
